# Org Hierarchy Refinement — Step-by-Step Test Notebook

This notebook tests the refinement agent step by step:
1. **Setup** — Configuration, pipeline init
2. **Load graph** — Pull existing graph from FalkorDB
3. **Inspect** — Review graph before refinement
4. **Test single node** — Refine one org unit manually
5. **Top-down pass** — Refine org units root to leaves
6. **Bottom-up pass** — Refine org units leaves to root
7. **Bidirectional merge** — Merge both perspectives
8. **Activity refinement** — Refine individual activity nodes
9. **Vector indexes** — Create/verify FalkorDB indexes
10. **Verify FalkorDB** — Check synced data
11. **Semantic search** — Test search on refined embeddings
12. **Export** — Save refinements to JSON

---
## Step 1: Configuration & Setup

In [1]:
# =============================================================================
# CONFIGURATION
# =============================================================================

import os
import sys
from dotenv import load_dotenv

sys.path.insert(0, '.')

load_dotenv()

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

# Data
DATA_SOURCE_PATH = os.getenv("DATA_SOURCE_PATH", "./data/samples/CASP_Activities_EN.xlsx")

# Graph structure
ACTIVITIES_AS_NODES = True

# Stores
VECTOR_STORE_TYPE = os.getenv("VECTOR_STORE_TYPE", "mock")
GRAPH_STORE_TYPE = "falkordb"
FALKORDB_URL = os.getenv("FALKORDB_URL", "")
FALKORDB_GRAPH_NAME = os.getenv("FALKORDB_GRAPH_NAME", "org_hierarchy")
GRAPH_LOCAL_PATH = os.getenv("GRAPH_LOCAL_PATH", "./graph_data/organization_graph.json")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "all-MiniLM-L6-v2")

# Refinement settings
REFINEMENT_STRATEGY = "bidirectional"   # "top_down", "bottom_up", "bidirectional"
USE_MOCK_LLM = False                    # True = test without API key
MAX_NODES = None                        # Set to e.g. 3 for quick testing

print("Configuration:")
print(f"  Data source:     {DATA_SOURCE_PATH}")
print(f"  Graph store:     {GRAPH_STORE_TYPE}")
print(f"  Strategy:        {REFINEMENT_STRATEGY}")
print(f"  Mock LLM:        {USE_MOCK_LLM}")
print(f"  Max nodes:       {MAX_NODES or 'all'}")
masked = FALKORDB_URL.split("@")[-1] if "@" in FALKORDB_URL else "(not set)"
print(f"  FalkorDB host:   {masked}")

Configuration:
  Data source:     ./data/samples/CASP_Activities_EN.xlsx
  Graph store:     falkordb
  Strategy:        bidirectional
  Mock LLM:        False
  Max nodes:       all
  FalkorDB host:   r-6jissuruar.instance-zyadijt4h.hc-v8noonp0c.europe-west1.gcp.f2e0a955bb84.cloud:55831


---
## Step 2: Create Pipeline & Load Graph from FalkorDB

In [2]:
# =============================================================================
# CREATE PIPELINE & INITIALIZE STORES
# =============================================================================

from pipeline import create_pipeline

pipeline = create_pipeline(
    data_source_path=DATA_SOURCE_PATH,
    activities_as_nodes=ACTIVITIES_AS_NODES,
    vector_store_type=VECTOR_STORE_TYPE,
    graph_store_type=GRAPH_STORE_TYPE,
    falkordb_url=FALKORDB_URL,
    verbose=True
)

# Initialize stores (needed when running steps manually)
pipeline._init_stores()

print(f"\n✓ Pipeline created")
print(f"  Vector store: {type(pipeline.vector_store).__name__}")
print(f"  Graph store:  {type(pipeline.graph_store).__name__}")
print(f"  Connected:    {getattr(pipeline.graph_store, '_connected', 'N/A')}")

✓ VectorStore: Mock (in-memory)
✓ GraphStore: FalkorDB (org_hierarchy)

✓ Pipeline created
  Vector store: MockVectorStore
  Graph store:  FalkorDBGraphStore
  Connected:    True


In [3]:
# =============================================================================
# LOAD EXISTING GRAPH FROM FALKORDB INTO LOCAL NETWORKX
# =============================================================================
import networkx as nx

graph = nx.DiGraph()

# Query nodes via raw FalkorDB graph to avoid the query() wrapper issue
# with Node objects containing vector lists
raw = pipeline.graph_store._graph

# --- Load org unit nodes (DG + OrganizationalUnit) ---
org_result = raw.query(
    "MATCH (n) WHERE n.node_type <> 'activity' "
    "RETURN n.node_id as nid, n.name as name, n.node_type as node_type, "
    "n.level as level, n.type as type, n.description as description, "
    "n.refined_description as refined_desc, "
    "n.refined_activity_summary as refined_summary, "
    "labels(n)[0] as label ORDER BY n.node_id"
)
for row in org_result.result_set:
    nid, name, node_type, level, ntype, desc, rdesc, rsummary, label = row
    if not nid:
        continue
    props = {"name": name or "", "node_type": node_type or "organization",
             "level": level if level is not None else 0,
             "type": ntype or "org_unit"}
    if desc:
        props["description"] = desc
    if rdesc:
        props["refined_description"] = rdesc
    if rsummary:
        props["refined_activity_summary"] = rsummary
    graph.add_node(nid, **props)

org_count = graph.number_of_nodes()
print(f"Loaded {org_count} org unit nodes")

# --- Load activity nodes ---
act_result = raw.query(
    "MATCH (n:Activity) "
    "RETURN n.node_id as nid, n.description as description, "
    "n.weight_pct as weight_pct, n.parent_org as parent_org, "
    "n.excel_row as excel_row, n.node_type as node_type, "
    "n.type as type, n.refined_description as rdesc, "
    "n.activity_keywords as keywords ORDER BY n.node_id"
)
for row in act_result.result_set:
    nid, desc, weight, parent_org, erow, node_type, ntype, rdesc, kw = row
    if not nid:
        continue
    props = {"description": desc or "", "weight_pct": weight or 0,
             "parent_org": parent_org or "", "excel_row": erow or 0,
             "node_type": node_type or "activity",
             "type": ntype or "activity"}
    if rdesc:
        props["refined_description"] = rdesc
    if kw:
        props["activity_keywords"] = kw
    graph.add_node(nid, **props)

act_count = graph.number_of_nodes() - org_count
print(f"Loaded {act_count} activity nodes")

# --- Load edges ---
edge_result = raw.query(
    "MATCH (a)-[r]->(b) "
    "RETURN a.node_id as src, b.node_id as tgt, type(r) as etype"
)
for row in edge_result.result_set:
    src, tgt, etype = row
    if src and tgt:
        graph.add_edge(src, tgt, edge_type=etype or "HAS_CHILD")

print(f"\n✓ Graph loaded from FalkorDB: {graph.number_of_nodes()} nodes "
      f"({org_count} org, {act_count} activities), "
      f"{graph.number_of_edges()} edges")

# Show org hierarchy
for nid in sorted(n for n, d in graph.nodes(data=True) if d.get("type") == "org_unit"):
    data = graph.nodes[nid]
    level = data.get("level", "?")
    name = data.get("name", "")[:50]
    print(f"  [{nid:10s}] L{level}  {name}")

Loaded 14 org unit nodes
Loaded 78 activity nodes

✓ Graph loaded from FalkorDB: 92 nodes (14 org, 78 activities), 91 edges
  [22        ] L0  Directorate-General for Cohesion, Agriculture and 
  [22-10     ] L0  Resources Unit
  [22A       ] L0  Directorate for Regional Development, Agriculture 
  [22A10     ] L0  Policy Department
  [22A20     ] L0  Secretariat of the Committee on Regional Developme
  [22A30     ] L0  Secretariat of the Committee on Agriculture and Ru
  [22A40     ] L0  Secretariat of the Committee on Fisheries
  [22B       ] L0  Directorate for Transport, Employment and Social A
  [22B00     ] L0  (synthesized) 22B00
  [22B0040   ] L0  Secretariat of the Special Committee on the Housin
  [22B10     ] L0  Policy Department
  [22B20     ] L0  Secretariat of the Committee on Transport and Tour
  [22B30     ] L0  Secretariat of the Committee on Employment and Soc
  [ORG_MASTER] L-1  European Parliament Organizations


---
## Step 3: Inspect Graph Before Refinement

In [4]:
# =============================================================================
# INSPECT: ORG UNITS BEFORE REFINEMENT
# =============================================================================

from refinement_agent import get_org_units, get_activity_nodes, get_activities, get_children

print("=" * 70)
print("ORG UNITS BEFORE REFINEMENT")
print("=" * 70)

for node_id in sorted(get_org_units(graph)):
    data = graph.nodes[node_id]
    activities = get_activities(graph, node_id)
    sub_orgs = get_children(graph, node_id, include_activities=False)
    has_refined = bool(data.get("refined_description"))

    print(f"\n[{node_id:10s}] {data.get('name', '')}")
    print(f"  Sub-orgs: {len(sub_orgs)} | Activities: {len(activities)} | Refined: {has_refined}")
    for act_id in activities[:3]:
        act = graph.nodes[act_id]
        print(f"    {act_id} ({act.get('weight_pct',0)}%) {act.get('description','')[:60]}...")
    if len(activities) > 3:
        print(f"    ... +{len(activities)-3} more activities")

print(f"\nTotal: {len(get_org_units(graph))} org units, {len(get_activity_nodes(graph))} activities")

ORG UNITS BEFORE REFINEMENT

[22        ] Directorate-General for Cohesion, Agriculture and Social Policies
  Sub-orgs: 3 | Activities: 7 | Refined: True
    22.7 (5%) Perform the functions of delegated authorizing officer....
    22.6 (5%) Coordinate the procedures of the general management and the ...
    22.5 (10%) Represent general management in interinstitutional relations...
    ... +4 more activities

[22-10     ] Resources Unit
  Sub-orgs: 0 | Activities: 8 | Refined: True
    22-10.15 (7%) Coordinate internal communication (writing and publication);...
    22-10.14 (8%) Management of the budget envelope for missions (3L and H3L),...
    22-10.13 (10%) Coordinate, establish and draft the general management budge...
    ... +5 more activities

[22A       ] Directorate for Regional Development, Agriculture and Fisheries
  Sub-orgs: 4 | Activities: 6 | Refined: False
    22A.21 (5%) Represent the institution in various internal committees and...
    22A.20 (5%) Ensure project mana

---
## Step 4: Create LLM & Refinement Agent

In [5]:
# =============================================================================
# CREATE LLM
# =============================================================================

from llm import create_llm

llm = create_llm(use_mock=USE_MOCK_LLM)

C:\Users\marci\anaconda3\envs\COURSERA_GENAI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ LLM: claude-sonnet-4-20250514


In [6]:
# =============================================================================
# CREATE REFINEMENT AGENT
# =============================================================================

from refinement_agent import create_refinement_agent

agent = create_refinement_agent(
    graph=graph,
    graph_store=pipeline.graph_store,
    vector_store=pipeline.vector_store,
    llm=llm,
    strategy=REFINEMENT_STRATEGY,
    refine_org_units=True,
    refine_activities=True,
    sync_to_falkordb=True,
    update_embeddings=True,
    create_vector_index=True,
    embedding_dimension=384,
    verbose=True,
    max_nodes=MAX_NODES
)

print(f"\n✓ Refinement agent created")
print(f"  Strategy:          {agent.config.strategy}")
print(f"  Refine org units:  {agent.config.refine_org_units}")
print(f"  Refine activities: {agent.config.refine_activities}")
print(f"  Sync to FalkorDB:  {agent.config.sync_to_falkordb}")
print(f"  Update embeddings: {agent.config.update_embeddings}")


✓ Refinement agent created
  Strategy:          bidirectional
  Refine org units:  True
  Refine activities: True
  Sync to FalkorDB:  True
  Update embeddings: True


---
## Step 5: Test Single Org Unit Refinement

Test refining one node to check the prompt, LLM response, and FalkorDB sync before running the full pipeline.

In [7]:
# =============================================================================
# TEST: REFINE A SINGLE ORG UNIT
# =============================================================================

from refinement_agent import (
    get_hierarchical_context, format_activities_for_org,
    parse_org_response, ORG_UNIT_REFINEMENT_PROMPT, SYSTEM_PROMPT
)

# Pick a test node dynamically — prefer a leaf unit that has activities
test_node = None
for nid in sorted(get_org_units(graph)):
    children = get_children(graph, nid, include_activities=False)
    activities = get_activities(graph, nid)
    if len(children) == 0 and len(activities) > 0:
        test_node = nid
        break

# Fallback: first org unit with activities
if test_node is None:
    for nid in sorted(get_org_units(graph)):
        if len(get_activities(graph, nid)) > 0:
            test_node = nid
            break

assert test_node is not None, "No org units with activities found in graph!"

data = graph.nodes[test_node]
context = get_hierarchical_context(graph, test_node)
activities_text = format_activities_for_org(graph, test_node)

print("=" * 70)
print(f"TEST NODE: [{test_node}] {data.get('name', '')}")
print("=" * 70)

print(f"\n--- Hierarchical Context ---")
print(f"  Parent:   {context['parent_info'][:80]}")
print(f"  Children: {context['children_info'][:80]}")
print(f"  Siblings: {context['siblings_info'][:80]}")

print(f"\n--- Activities ({len(get_activities(graph, test_node))}) ---")
print(activities_text[:500])

TEST NODE: [22-10] Resources Unit

--- Hierarchical Context ---
  Parent:   Directorate-General for Cohesion, Agriculture and Social Policies
  Context: The
  Children: LEAF UNIT — No subordinate organizational units
  Siblings: Directorate for Transport, Emp, Directorate for Regional Devel

--- Activities (8) ---
Activity 1 [22-10.10] — 15% of workload:
  Develop the strategic planning and reporting cycle for general management, coordinate risk management and the business continuity plan. Ensure the representation of general management in various horizontal work groups; management and coordination of the DG's EMAS actions; ensure the coordination of 'simplification' actions.

Activity 2 [22-10.11] — 10% of workload:
  Organize internal training in cooperation with DG PERS services (training plan, budget 


In [8]:
# =============================================================================
# TEST: VIEW FULL PROMPT (what the LLM sees)
# =============================================================================

prompt = ORG_UNIT_REFINEMENT_PROMPT.format(
    name=data.get("name", test_node),
    code=test_node,
    level=data.get("level", 0),
    activities_detail=activities_text,
    parent_info=context["parent_info"],
    children_info=context["children_info"],
    siblings_info=context["siblings_info"]
)

print("=" * 70)
print("FULL PROMPT SENT TO LLM")
print("=" * 70)
print(prompt[:2000])
if len(prompt) > 2000:
    print(f"\n... ({len(prompt)} chars total)")

FULL PROMPT SENT TO LLM
Analyze this organizational unit and create an augmented description.

═══════════════════════════════════════════════════════════════════════════════
UNIT INFORMATION
═══════════════════════════════════════════════════════════════════════════════
Unit Name: Resources Unit
Unit Code: 22-10
Hierarchy Level: 0

═══════════════════════════════════════════════════════════════════════════════
ACTIVITIES ASSIGNED TO THIS UNIT
═══════════════════════════════════════════════════════════════════════════════
Activity 1 [22-10.10] — 15% of workload:
  Develop the strategic planning and reporting cycle for general management, coordinate risk management and the business continuity plan. Ensure the representation of general management in various horizontal work groups; management and coordination of the DG's EMAS actions; ensure the coordination of 'simplification' actions.

Activity 2 [22-10.11] — 10% of workload:
  Organize internal training in cooperation with DG PERS serv

In [9]:
# =============================================================================
# TEST: CALL LLM AND PARSE RESPONSE
# =============================================================================

response = llm.generate(prompt, SYSTEM_PROMPT)
result = parse_org_response(response)

print("=" * 70)
print("LLM RESPONSE (parsed)")
print("=" * 70)
print(f"\nREFINED_DESCRIPTION:")
print(result["refined_description"][:500])
print(f"\nREFINED_ACTIVITY_SUMMARY:")
print(result["refined_activity_summary"][:500])

LLM RESPONSE (parsed)

REFINED_DESCRIPTION:
The Resources Unit (22-10) serves as the comprehensive administrative backbone of the Directorate-General for Cohesion, Agriculture and Social Policies, orchestrating essential support functions that enable the DG's strategic operations within the European Parliament. Operating as a leaf unit at the foundational organizational level, it manages two primary operational domains: human resources management (20% allocation for staff recruitment, organizational structure, and personnel administration

REFINED_ACTIVITY_SUMMARY:
• Human Resources & Personnel Management (20%): Staff recruitment, organizational chart management, mobility exercises, intern selection, rights and obligations administration, strategic HR analysis and advisory services
• Financial Management & Budget Oversight (30%): Centralized financial systems management (20%), budget coordination and monitoring (10%) including FMS/Webcontracts systems, budgetary transfers, audit coordi

In [10]:
# =============================================================================
# TEST: REFINE VIA AGENT (includes FalkorDB sync + embedding)
# =============================================================================

agent_result = agent.refine_org_unit(test_node)

print("=" * 70)
print(f"AGENT RESULT for {test_node}")
print("=" * 70)
print(f"\nDescription: {agent_result['refined_description'][:200]}...")
print(f"\nSummary: {agent_result['refined_activity_summary'][:200]}...")

# Verify in local graph
local = graph.nodes[test_node]
print(f"\n✓ Local graph updated: {bool(local.get('refined_description'))}")

# Verify in FalkorDB
try:
    fb_check = pipeline.graph_store.query(
        "MATCH (n {node_id: $id}) RETURN n.refined_description as desc",
        {"id": test_node}
    )
    print(f"✓ FalkorDB synced: {bool(fb_check and fb_check[0].get('desc'))}")
except Exception as e:
    print(f"⚠ FalkorDB check: {e}")

AGENT RESULT for 22-10

Description: The Resources Unit (22-10) serves as the comprehensive administrative backbone for the Directorate-General for Cohesion, Agriculture and Social Policies, providing essential support services across hu...

Summary: • Human Resources & Staff Management (20%): Comprehensive staff recruitment, organizational chart management, mobility exercises, intern selection, and rating exercise coordination
• Financial Managem...

✓ Local graph updated: True
✓ FalkorDB synced: True


---
## Step 6: Test Single Activity Refinement

In [11]:
# =============================================================================
# TEST: REFINE A SINGLE ACTIVITY
# =============================================================================

from refinement_agent import get_activity_context, ACTIVITY_REFINEMENT_PROMPT, parse_activity_response

# Pick first activity of our test org unit
test_activities = get_activities(graph, test_node)
test_act = test_activities[0] if test_activities else None

if test_act:
    act_data = graph.nodes[test_act]
    act_context = get_activity_context(graph, test_act)

    print("=" * 70)
    print(f"TEST ACTIVITY: [{test_act}]")
    print("=" * 70)
    print(f"  Weight: {act_data.get('weight_pct', 0)}%")
    print(f"  Parent org: {act_context['parent_org_name']}")
    print(f"  Original: {act_data.get('description', '')[:150]}...")

    # Refine via agent
    act_result = agent.refine_activity(test_act)

    print(f"\n  Refined: {act_result['refined_description'][:150]}...")
    print(f"  Keywords: {act_result['activity_keywords']}")

    # Verify FalkorDB sync
    try:
        fb_act = pipeline.graph_store.query(
            "MATCH (n {node_id: $id}) RETURN n.refined_description as desc, n.activity_keywords as kw",
            {"id": test_act}
        )
        if fb_act:
            print(f"\n✓ FalkorDB synced: desc={bool(fb_act[0].get('desc'))}, keywords={bool(fb_act[0].get('kw'))}")
    except Exception as e:
        print(f"⚠ FalkorDB check: {e}")
else:
    print("No activities found for test node")

TEST ACTIVITY: [22-10.15]
  Weight: 7%
  Parent org: Resources Unit
  Original: Coordinate internal communication (writing and publication); organize internal events; manage the DG website; communication services to other units; e...

  Refined: This activity encompasses comprehensive internal communication management and digital presence coordination for the Directorate-General for Cohesion, ...
  Keywords: internal communication, content writing, publication management, event organization, website management, communication services, visual identity compliance, brand consistency, digital content, editorial coordination

✓ FalkorDB synced: desc=True, keywords=True


---
## Step 7: Run Top-Down Refinement (all org units)

In [12]:
# =============================================================================
# TOP-DOWN REFINEMENT
# =============================================================================

# Clear any previous refinements first
for n in get_org_units(graph):
    graph.nodes[n]["refined_description"] = ""
    graph.nodes[n]["refined_activity_summary"] = ""

td_results = agent.run_top_down(max_nodes=MAX_NODES)


══════════════════════════════════════════════════════════════════════
TOP-DOWN REFINEMENT (Root → Leaves)
══════════════════════════════════════════════════════════════════════
Processing 14 org units...

[  1/14] ORG_MASTER European Parliament Organizations
           → The European Parliament Organizations unit (ORG_MASTER) serv...

[  2/14] 22         Directorate-General for Cohesion, Agricu
           → The Directorate-General for Cohesion, Agriculture and Social...

[  3/14] 22B        Directorate for Transport, Employment an
           → The Directorate for Transport, Employment and Social Affairs...

[  4/14] 22A        Directorate for Regional Development, Ag
           → The Directorate for Regional Development, Agriculture and Fi...

[  5/14] 22-10      Resources Unit
           → The Resources Unit (22-10) serves as the comprehensive admin...

[  6/14] 22B30      Secretariat of the Committee on Employme
           → The Secretariat of the Committee on Employment and Social

In [13]:
# =============================================================================
# INSPECT TOP-DOWN RESULTS
# =============================================================================

print("=" * 70)
print("TOP-DOWN RESULTS")
print("=" * 70)

for node_id, result in sorted(td_results.items()):
    name = graph.nodes[node_id].get("name", "")[:40]
    desc = result["refined_description"][:80]
    print(f"\n[{node_id:10s}] {name}")
    print(f"  -> {desc}...")

TOP-DOWN RESULTS

[22        ] Directorate-General for Cohesion, Agricu
  -> The Directorate-General for Cohesion, Agriculture and Social Policies (Unit 22) ...

[22-10     ] Resources Unit
  -> The Resources Unit (22-10) serves as the comprehensive administrative backbone f...

[22A       ] Directorate for Regional Development, Ag
  -> The Directorate for Regional Development, Agriculture and Fisheries (22A) serves...

[22A10     ] Policy Department
  -> The Policy Department (22A10) serves as a specialized research and analytical un...

[22A20     ] Secretariat of the Committee on Regional
  -> The Secretariat of the Committee on Regional Development (22A20) serves as the p...

[22A30     ] Secretariat of the Committee on Agricult
  -> The Secretariat of the Committee on Agriculture and Rural Development (22A30) se...

[22A40     ] Secretariat of the Committee on Fisherie
  -> The Secretariat of the Committee on Fisheries (22A40) serves as the primary admi...

[22B       ] Directorat

---
## Step 8: Run Bottom-Up Refinement

In [14]:
# =============================================================================
# SNAPSHOT TOP-DOWN RESULTS, THEN RUN BOTTOM-UP
# =============================================================================

agent.td_snapshot = {}
for node_id in get_org_units(graph):
    data = graph.nodes[node_id]
    agent.td_snapshot[node_id] = {
        "refined_description": data.get("refined_description", ""),
        "refined_activity_summary": data.get("refined_activity_summary", "")
    }

print(f"✓ Snapshotted {len(agent.td_snapshot)} top-down results")

bu_results = agent.run_bottom_up(max_nodes=MAX_NODES)

✓ Snapshotted 14 top-down results

══════════════════════════════════════════════════════════════════════
BOTTOM-UP REFINEMENT (Leaves → Root)
══════════════════════════════════════════════════════════════════════
Processing 14 org units...

[  1/14] 22B0040    Secretariat of the Special Committee on 
           → The Secretariat of the Special Committee on the Housing Cris...

[  2/14] 22A10      Policy Department
           → The Policy Department (22A10) serves as the specialized rese...

[  3/14] 22A20      Secretariat of the Committee on Regional
           → The Secretariat of the Committee on Regional Development (22...

[  4/14] 22A30      Secretariat of the Committee on Agricult
           → The Secretariat of the Committee on Agriculture and Rural De...

[  5/14] 22A40      Secretariat of the Committee on Fisherie
           → The Secretariat of the Committee on Fisheries (22A40) serves...

[  6/14] 22B00      (synthesized) 22B00
           → Unit 22B00 operates as a synthesi

In [15]:
# =============================================================================
# COMPARE TOP-DOWN vs BOTTOM-UP
# =============================================================================

print("=" * 70)
print("TOP-DOWN vs BOTTOM-UP COMPARISON")
print("=" * 70)

for node_id in sorted(get_org_units(graph)):
    td = agent.td_snapshot.get(node_id, {}).get("refined_description", "")[:80]
    bu = bu_results.get(node_id, {}).get("refined_description", "")[:80]

    if td or bu:
        name = graph.nodes[node_id].get("name", "")[:30]
        print(f"\n[{node_id}] {name}")
        print(f"  TD: {td}...")
        print(f"  BU: {bu}...")

TOP-DOWN vs BOTTOM-UP COMPARISON

[22] Directorate-General for Cohesi
  TD: The Directorate-General for Cohesion, Agriculture and Social Policies (Unit 22) ...
  BU: The Directorate-General for Cohesion, Agriculture and Social Policies (22) serve...

[22-10] Resources Unit
  TD: The Resources Unit (22-10) serves as the comprehensive administrative backbone f...
  BU: The Resources Unit (22-10) serves as the comprehensive operational backbone for ...

[22A] Directorate for Regional Devel
  TD: The Directorate for Regional Development, Agriculture and Fisheries (22A) serves...
  BU: The Directorate for Regional Development, Agriculture and Fisheries (22A) serves...

[22A10] Policy Department
  TD: The Policy Department (22A10) serves as a specialized research and analytical un...
  BU: The Policy Department (22A10) serves as the specialized research and analytical ...

[22A20] Secretariat of the Committee o
  TD: The Secretariat of the Committee on Regional Development (22A20) serves as 

---
## Step 9: Bidirectional Merge

In [16]:
# =============================================================================
# MERGE TOP-DOWN + BOTTOM-UP
# =============================================================================

merged_count = agent.run_bidirectional_merge(td_results, bu_results)

print(f"\n✓ Merged {merged_count} org units")


══════════════════════════════════════════════════════════════════════
BIDIRECTIONAL MERGE
══════════════════════════════════════════════════════════════════════
[MERGED] 22 Directorate-General for Cohesion, A
[MERGED] 22-10 Resources Unit
[MERGED] 22A Directorate for Regional Developmen
[MERGED] 22A10 Policy Department
[MERGED] 22A20 Secretariat of the Committee on Reg
[MERGED] 22A30 Secretariat of the Committee on Agr
[MERGED] 22A40 Secretariat of the Committee on Fis
[MERGED] 22B Directorate for Transport, Employme
[MERGED] 22B00 (synthesized) 22B00
[MERGED] 22B0040 Secretariat of the Special Committe
[MERGED] 22B10 Policy Department
[MERGED] 22B20 Secretariat of the Committee on Tra
[MERGED] 22B30 Secretariat of the Committee on Emp
[MERGED] ORG_MASTER European Parliament Organizations

✓ Merged 14 org units

✓ Merged 14 org units


In [17]:
# =============================================================================
# INSPECT FINAL MERGED DESCRIPTIONS
# =============================================================================

print("=" * 70)
print("FINAL MERGED DESCRIPTIONS")
print("=" * 70)

for node_id in sorted(get_org_units(graph)):
    data = graph.nodes[node_id]
    desc = data.get("refined_description", "")
    summary = data.get("refined_activity_summary", "")

    if desc:
        print(f"\n[{node_id}] {data.get('name', '')}")
        print(f"  Description: {desc[:200]}...")
        if summary:
            print(f"  Summary: {summary[:200]}...")

FINAL MERGED DESCRIPTIONS

[22] Directorate-General for Cohesion, Agriculture and Social Policies
  Description: The Directorate-General for Cohesion, Agriculture and Social Policies (Unit 22) operates as a cornerstone strategic coordination body within the European Parliament's organizational framework, serving...
  Summary: • Strategic Leadership & Parliamentary Reform Implementation (25%): Define institutional strategy, develop general management functions, strengthen Parliament's legislative/budgetary/control powers th...

[22-10] Resources Unit
  Description: The Resources Unit (22-10) functions as the comprehensive administrative and strategic backbone for the Directorate-General for Cohesion, Agriculture and Social Policies within the European Parliament...
  Summary: • Human Resources & Staff Management (20%): Complete recruitment management, organizational chart oversight, staff rights and obligations administration, performance evaluations, rating exercises, and...

[22A] Dir

---
## Step 10: Refine All Activities

In [18]:
# =============================================================================
# REFINE ALL ACTIVITIES
# =============================================================================

activity_results = agent.refine_all_activities(max_nodes=MAX_NODES)


══════════════════════════════════════════════════════════════════════
ACTIVITY REFINEMENT
══════════════════════════════════════════════════════════════════════
Processing 78 activities...

[  1/78] 22-10.10             (22-10, 15%)
           → This activity encompasses the comprehensive development and ...

[  2/78] 22-10.11             (22-10, 10%)
           → This activity encompasses comprehensive human resource devel...

[  3/78] 22-10.12             (22-10, 10%)
           → This logistics coordination activity (representing 10% of Re...

[  4/78] 22-10.13             (22-10, 10%)
           → This activity encompasses comprehensive budget management an...

[  5/78] 22-10.14             (22-10, 8%)
           → This activity encompasses comprehensive mission budget admin...

[  6/78] 22-10.15             (22-10, 7%)
           → This activity encompasses comprehensive internal and externa...

[  7/78] 22-10.8              (22-10, 20%)
           → This human resources managem

In [19]:
# =============================================================================
# INSPECT REFINED ACTIVITIES (sample)
# =============================================================================

print("=" * 70)
print("REFINED ACTIVITIES (first 10)")
print("=" * 70)

for act_id in sorted(activity_results.keys())[:10]:
    data = graph.nodes[act_id]
    result = activity_results[act_id]

    print(f"\n[{act_id}] ({data.get('weight_pct', 0)}%) parent: {data.get('parent_org', '?')}")
    print(f"  Original: {data.get('description', '')[:80]}...")
    print(f"  Refined:  {result['refined_description'][:80]}...")
    print(f"  Keywords: {result['activity_keywords'][:60]}")

REFINED ACTIVITIES (first 10)

[22-10.10] (15%) parent: 22-10
  Original: Develop the strategic planning and reporting cycle for general management, coord...
  Refined:  This activity encompasses the comprehensive development and maintenance of strat...
  Keywords: strategic planning, reporting cycles, risk management, busin

[22-10.11] (10%) parent: 22-10
  Original: Organize internal training in cooperation with DG PERS services (training plan, ...
  Refined:  This activity encompasses comprehensive human resource development and training ...
  Keywords: training coordination, human resource development, DG PERS c

[22-10.12] (10%) parent: 22-10
  Original: Logistics: Ensure the logistical support necessary for the proper functioning of...
  Refined:  This logistics coordination activity (representing 10% of Resources Unit workloa...
  Keywords: logistics coordination, facility management, space planning,

[22-10.13] (10%) parent: 22-10
  Original: Coordinate, establish and draft the

---
## Step 11: Setup Vector Indexes

In [20]:
# =============================================================================
# CREATE VECTOR INDEXES IN FALKORDB
# =============================================================================

agent.setup_vector_indexes()


══════════════════════════════════════════════════════════════════════
CREATING VECTOR INDEXES
══════════════════════════════════════════════════════════════════════
⚠ Vector index creation: errMsg: Invalid input 'o': expected CREATE INDEX FOR line: 2, column: 33, offset: 33 errCtx:             CREATE VECTOR INDEX org_refined_embedding_idx errCtxOffset: 32
⚠ Vector index creation: errMsg: Invalid input 'd': expected CREATE INDEX FOR line: 2, column: 33, offset: 33 errCtx:             CREATE VECTOR INDEX dg_refined_embedding_idx errCtxOffset: 32
⚠ Vector index creation: errMsg: Invalid input 'a': expected CREATE INDEX FOR line: 2, column: 33, offset: 33 errCtx:             CREATE VECTOR INDEX activity_refined_embedding_idx errCtxOffset: 32
✓ Vector indexes created for OrganizationalUnit, DG, and Activity


---
## Step 12: Verify FalkorDB

In [21]:
# =============================================================================
# VERIFY REFINEMENTS IN FALKORDB
# =============================================================================

print("=" * 70)
print("FALKORDB VERIFICATION")
print("=" * 70)

try:
    # Refined org units — use scalar returns to avoid Node object issues
    refined_orgs = pipeline.graph_store.query(
        "MATCH (n) WHERE n.refined_description IS NOT NULL "
        "AND n.node_type <> 'activity' "
        "RETURN n.node_id as id, n.name as name, "
        "substring(n.refined_description, 0, 80) as desc "
        "ORDER BY n.node_id"
    )
    print(f"\nRefined org units: {len(refined_orgs)}")
    for row in refined_orgs:
        print(f"  [{row.get('id')}] {row.get('desc')}...")

    # Refined activities
    refined_acts = pipeline.graph_store.query(
        "MATCH (n:Activity) WHERE n.refined_description IS NOT NULL "
        "RETURN count(n) as count"
    )
    print(f"\nRefined activities: {refined_acts[0].get('count', 0) if refined_acts else 0}")

    # Embeddings
    emb_count = pipeline.graph_store.query(
        "MATCH (n) WHERE n.refined_embedding IS NOT NULL "
        "RETURN count(n) as count"
    )
    print(f"Nodes with refined embeddings: {emb_count[0].get('count', 0) if emb_count else 0}")

    # Vector indexes
    print("\nVector indexes:")
    try:
        indexes = pipeline.graph_store._graph.query("CALL db.indexes()")
        for row in indexes.result_set:
            print(f"  {row}")
    except Exception:
        print("  (Could not list indexes)")

except Exception as e:
    print(f"  Error: {e}")

FALKORDB VERIFICATION

Refined org units: 25
  [22] The Directorate-General for Cohesion, Agriculture and Social Policies (Unit 22) ...
  [22-10] The Resources Unit (22-10) functions as the comprehensive administrative and str...
  [22A] The Directorate for Regional Development, Agriculture and Fisheries (22A) serves...
  [22A10] The Policy Department (22A10) operates as a specialized research and analytical ...
  [22A20] The Secretariat of the Committee on Regional Development (22A20) operates as the...
  [22A30] The Secretariat of the Committee on Agriculture and Rural Development (22A30) op...
  [22A40] The Secretariat of the Committee on Fisheries (22A40) serves as the comprehensiv...
  [22B] The Directorate for Transport, Employment and Social Affairs (Unit 22B) operates...
  [22B00] Unit 22B00 functions as the synthesized administrative headquarters and strategi...
  [22B0040] The Secretariat of the Special Committee on the Housing Crisis in the European U...
  [22B10] The Policy

In [22]:
# =============================================================================
# VERIFY: SAMPLE NODE IN FALKORDB (full detail)
# =============================================================================

sample_id = test_node

try:
    # Return scalar properties to avoid unhashable list errors
    result = pipeline.graph_store.query(
        "MATCH (n {node_id: $id}) "
        "RETURN n.node_id as node_id, n.name as name, "
        "n.node_type as node_type, n.level as level, "
        "n.refined_description as refined_description, "
        "n.refined_activity_summary as refined_activity_summary",
        {"id": sample_id}
    )
    if result:
        print(f"Node [{sample_id}] properties in FalkorDB:")
        for k, v in result[0].items():
            val = str(v)[:100] if v else "(empty)"
            print(f"  {k}: {val}")
    else:
        print(f"Node [{sample_id}] not found in FalkorDB")
except Exception as e:
    print(f"Error: {e}")

Node [22-10] properties in FalkorDB:
  node_id: 22-10
  name: Resources Unit
  node_type: organization
  level: (empty)
  refined_description: The Resources Unit (22-10) functions as the comprehensive administrative and strategic backbone for 
  refined_activity_summary: • Human Resources & Staff Management (20%): Complete recruitment management, organizational chart ov


---
## Step 13: Semantic Search on Refined Embeddings

In [23]:
# =============================================================================
# SEMANTIC SEARCH ON REFINED EMBEDDINGS
# =============================================================================

test_queries = [
    "budget management and financial oversight",
    "committee secretariat legislative support",
    "fisheries and agriculture policy development",
    "human resources recruitment and staff management",
    "transport and tourism policy coordination",
]

print("=" * 70)
print("SEMANTIC SEARCH — REFINED EMBEDDINGS")
print("=" * 70)

for query in test_queries:
    print(f"\nQuery: '{query}'")
    try:
        embedding = pipeline.vector_store.get_embedding(query)

        # Search org units
        org_results = pipeline.graph_store.query(
            "CALL db.idx.vector.queryNodes("
            "'OrganizationalUnit', 'refined_embedding', 3, vecf32($embedding)"
            ") YIELD node, score "
            "RETURN node.node_id as id, node.name as name, score "
            "ORDER BY score DESC",
            {"embedding": embedding}
        )
        if org_results:
            print("  Org matches:")
            for row in org_results:
                print(f"    [{row.get('id')}] (score: {row.get('score',0):.3f}) {row.get('name')}")

        # Search DG nodes too
        dg_results = pipeline.graph_store.query(
            "CALL db.idx.vector.queryNodes("
            "'DG', 'refined_embedding', 3, vecf32($embedding)"
            ") YIELD node, score "
            "RETURN node.node_id as id, node.name as name, score "
            "ORDER BY score DESC",
            {"embedding": embedding}
        )
        if dg_results:
            print("  DG matches:")
            for row in dg_results:
                print(f"    [{row.get('id')}] (score: {row.get('score',0):.3f}) {row.get('name')}")

        # Search activities
        act_results = pipeline.graph_store.query(
            "CALL db.idx.vector.queryNodes("
            "'Activity', 'refined_embedding', 3, vecf32($embedding)"
            ") YIELD node, score "
            "RETURN node.node_id as id, score, "
            "substring(node.refined_description, 0, 60) as desc "
            "ORDER BY score DESC",
            {"embedding": embedding}
        )
        if act_results:
            print("  Activity matches:")
            for row in act_results:
                print(f"    [{row.get('id')}] (score: {row.get('score',0):.3f}) {row.get('desc')}...")

    except Exception as e:
        print(f"  Error: {e}")

SEMANTIC SEARCH — REFINED EMBEDDINGS

Query: 'budget management and financial oversight'
Query error: Invalid arguments for procedure 'db.idx.vector.queryNodes'
Query error: Invalid arguments for procedure 'db.idx.vector.queryNodes'
Query error: Invalid arguments for procedure 'db.idx.vector.queryNodes'

Query: 'committee secretariat legislative support'
Query error: Invalid arguments for procedure 'db.idx.vector.queryNodes'
Query error: Invalid arguments for procedure 'db.idx.vector.queryNodes'
Query error: Invalid arguments for procedure 'db.idx.vector.queryNodes'

Query: 'fisheries and agriculture policy development'
Query error: Invalid arguments for procedure 'db.idx.vector.queryNodes'
Query error: Invalid arguments for procedure 'db.idx.vector.queryNodes'
Query error: Invalid arguments for procedure 'db.idx.vector.queryNodes'

Query: 'human resources recruitment and staff management'
Query error: Invalid arguments for procedure 'db.idx.vector.queryNodes'
Query error: Invalid argu

---
## Step 14: Export Refinements

In [24]:
# =============================================================================
# EXPORT REFINEMENTS TO JSON
# =============================================================================

export_path = agent.export_refinements("./output/refinements.json")

✓ Exported refinements: ./output/refinements.json (14 org units, 78 activities)


---
## Summary

In [25]:
# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 70)
print("REFINEMENT SUMMARY")
print("=" * 70)

org_units = get_org_units(graph)
activities = get_activity_nodes(graph)

org_refined = sum(1 for n in org_units if graph.nodes[n].get("refined_description"))
act_refined = sum(1 for n in activities if graph.nodes[n].get("refined_description"))
act_keywords = sum(1 for n in activities if graph.nodes[n].get("activity_keywords"))

print(f"  Strategy:              {REFINEMENT_STRATEGY}")
print(f"  LLM:                   {llm.model}")
print(f"  Mock mode:             {USE_MOCK_LLM}")
print(f"  Org units total:       {len(org_units)}")
print(f"  Org units refined:     {org_refined}")
print(f"  Activities total:      {len(activities)}")
print(f"  Activities refined:    {act_refined}")
print(f"  Activities w/keywords: {act_keywords}")
print(f"  FalkorDB synced:       {agent._stats.get('synced_to_falkordb', 0)}")
print(f"  Embeddings updated:    {agent._stats.get('embeddings_updated', 0)}")
print("\n✓ Refinement pipeline complete")


REFINEMENT SUMMARY
  Strategy:              bidirectional
  LLM:                   claude-sonnet-4-20250514
  Mock mode:             False
  Org units total:       14
  Org units refined:     14
  Activities total:      78
  Activities refined:    78
  Activities w/keywords: 78
  FalkorDB synced:       122
  Embeddings updated:    122

✓ Refinement pipeline complete
